# Cut-in Risk Analysis: Thesis Runner

This notebook runs the thesis pipeline in a simple and reproducible way.

## How To Use

1. Run setup and path check cells.
2. Choose one mode: `quick`, `full`, or `custom`.
3. Run the execution cell.
4. Check outputs in the final cells.

In [ ]:
from __future__ import annotations

import os
import shlex
import subprocess
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'pyproject.toml').exists():
            return candidate
    return start


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
os.chdir(PROJECT_ROOT)
print('Project root:', PROJECT_ROOT)

## Path Check

In [ ]:
from cutin_risk.paths import dataset_root_path, outputs_root_path, step14_codes_csv_path

dataset_root = dataset_root_path()
outputs_root = outputs_root_path()
codes_csv = step14_codes_csv_path()

print('dataset_root:', dataset_root)
print('outputs_root:', outputs_root)
print('step14_codes_csv:', codes_csv)

check = pd.DataFrame([
    {'item': 'dataset_root', 'path': str(dataset_root), 'exists': dataset_root.exists()},
    {'item': 'outputs_root', 'path': str(outputs_root), 'exists': outputs_root.exists()},
    {'item': 'step14_codes_csv', 'path': str(codes_csv), 'exists': codes_csv.exists()},
])
display(check)

## Run Mode

In [ ]:
# Choose one: 'quick', 'full', 'custom'
MODE = 'quick'

# Only used when MODE == 'custom'
CUSTOM_STEPS = ['step08', 'step09', 'step10']

# Step09 option
RECORDINGS_FOR_BATCH = '1-10'

# Safety switch (recommended): keep False until paths/settings are confirmed
RUN_PIPELINE = False

DRY_RUN = False
STOP_ON_ERROR = True

print('MODE:', MODE)
print('CUSTOM_STEPS:', CUSTOM_STEPS)
print('RECORDINGS_FOR_BATCH:', RECORDINGS_FOR_BATCH)
print('RUN_PIPELINE:', RUN_PIPELINE)
print('DRY_RUN:', DRY_RUN)
print('STOP_ON_ERROR:', STOP_ON_ERROR)

## Step List

In [6]:
STEP_COMMANDS = {
    'step01': [sys.executable, 'scripts/step01_recording_report.py'],
    'step02': [sys.executable, 'scripts/step02_lane_change_report.py'],
    'step03': [sys.executable, 'scripts/step03_cutin_report.py'],
    'step04': [sys.executable, 'scripts/step04_risk_metrics_report.py'],
    'step05': [sys.executable, 'scripts/step05_visualize_top_cutins.py'],
    'step06': [sys.executable, 'scripts/step06_neighbor_reconstruction_report.py'],
    'step07': [sys.executable, 'scripts/step07_xy_lane_pipeline_report.py'],
    'step08': [sys.executable, 'scripts/step08_stage_features.py', '--recording-id', '01'],
    'step09': [sys.executable, 'scripts/step09_batch_stage_features.py', '--recordings', RECORDINGS_FOR_BATCH],
    'step10': [sys.executable, 'scripts/step10_risk_report.py'],
    'step11': [sys.executable, 'scripts/step11_early_warning_rule_search.py'],
    'step12': [sys.executable, 'scripts/step12_early_warning_logreg.py'],
    'step13a': [sys.executable, 'scripts/step13a_logreg_tuned.py'],
    'step13b': [sys.executable, 'scripts/step13b_realistic_follower_warning.py'],
    'step14': [sys.executable, 'scripts/step14_sfc_binary_encode.py'],
    'step14r': [sys.executable, 'scripts/step14_sfc_binary_report.py'],
    'step15a': [sys.executable, 'scripts/step15a_sfc_mirror_normalize.py'],
    'step15b': [sys.executable, 'scripts/step15b_sfc_weighted_stage_features.py'],
    'step15c': [sys.executable, 'scripts/step15c_sfc_prediction.py'],
    'step16': [sys.executable, 'scripts/step16_sfc_predict_lanechange_cutin.py'],
}

QUICK_STEPS = ['step02', 'step03', 'step04', 'step08']
FULL_STEPS = list(STEP_COMMANDS.keys())

display(pd.DataFrame({'available_steps': list(STEP_COMMANDS.keys())}))

NameError: name 'sys' is not defined

## Run Pipeline

In [ ]:
def run_steps(step_ids: list[str]) -> None:
    for sid in step_ids:
        if sid not in STEP_COMMANDS:
            raise ValueError(f'Unknown step: {sid}')
        cmd = STEP_COMMANDS[sid]
        print('\n' + '=' * 20 + f' {sid} ' + '=' * 20)
        print('$', ' '.join(shlex.quote(x) for x in cmd))

        if DRY_RUN:
            print('[DRY_RUN] skipped')
            continue

        rc = subprocess.call(cmd, cwd=str(PROJECT_ROOT))
        if rc != 0:
            print(f'[ERROR] {sid} failed with exit code {rc}')
            if STOP_ON_ERROR:
                raise RuntimeError(f'Stopped at {sid}')


if MODE == 'quick':
    selected = QUICK_STEPS
elif MODE == 'full':
    selected = FULL_STEPS
elif MODE == 'custom':
    selected = CUSTOM_STEPS
else:
    raise ValueError("MODE must be 'quick', 'full', or 'custom'")

print('Selected steps:', selected)
if RUN_PIPELINE:
    run_steps(selected)
else:
    print('RUN_PIPELINE is False. Set RUN_PIPELINE = True to execute selected steps.')

## Output Check

In [ ]:
from cutin_risk.paths import outputs_root_path

out = outputs_root_path()
targets = {
    'step8_dir': out / 'reports/step8_recording_01',
    'step9_merged': out / 'reports/step9_batch/cutin_stage_features_merged.csv',
    'step10_dir': out / 'reports/step10_risk',
    'step14_codes': out / 'reports/step14_sfc_binary/sfc_binary_codes_long_hilbert.csv',
}

rows = [{'name': k, 'path': str(v), 'exists': v.exists()} for k, v in targets.items()]
display(pd.DataFrame(rows))

merged = targets['step9_merged']
if merged.exists():
    df = pd.read_csv(merged)
    print('Merged rows:', len(df), 'columns:', len(df.columns))
    display(df.head(5))

## SFC Decode Demo (3x3)

In [ ]:
H = [[0, 1, 14, 15], [3, 2, 13, 12], [4, 7, 8, 11], [5, 6, 9, 10]]

def code_to_3x3(code: int):
    return [[(int(code) >> H[r][c]) & 1 for c in range(3)] for r in range(3)]

csv_path = step14_codes_csv_path()
if csv_path.exists():
    one = pd.read_csv(csv_path, usecols=['event_id', 'frame', 'code', 'code_hex'], nrows=1).iloc[0]
    matrix = code_to_3x3(int(one['code']))
    print('event_id:', int(one['event_id']), 'frame:', int(one['frame']), 'code:', int(one['code']), 'hex:', one['code_hex'])
    print('\n'.join(' '.join(str(v) for v in row) for row in matrix))
else:
    print('File not found:', csv_path)